# Broadband Continuous Signal Processing with STFT-Based Propagation

This notebook demonstrates the BroadbandPassiveSonarArraySimulator which uses frequency-domain propagation for continuous broadband signals. It showcases:

- Continuous phase-coherent signal generation
- STFT-based frequency-domain propagation
- Time-varying transfer functions (moving platform)
- Overlap-add signal reconstruction

## Underwater Acoustics Background

In passive sonar, acoustic signals from underwater targets are received by sensor arrays. This notebook simulates:

1. **Source Signal Generation**: Realistic ship signatures with broadband tonals (propeller, machinery) and colored noise (cavitation, turbulence)
2. **Acoustic Propagation**: Frequency-domain ray tracing using RTRS (Range-dependent Two-dimensional Ray Simulation)
3. **Array Reception**: Multi-sensor time-series signals with proper phase relationships

**Key Concept - FFT Scaling**: When converting time-domain signals to frequency domain, the FFT introduces a scaling factor of N/2 (for single-sided spectrum). To get proper amplitude in µPa, we multiply by `2.0/N`. Similarly, spectrograms return Power Spectral Density (PSD) in µPa²/Hz, which must be properly referenced to 1 µPa for underwater acoustics.



## 1. Import Libraries and Set Random Seed

In [1]:
from datetime import datetime, timedelta

import numpy as np
import plotly.graph_objects as go
from IPython.display import Audio, display
from plotly.subplots import make_subplots
from scipy import signal as scipy_signal
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from nereus.models.environment import FlatBathymetry, Linear
from nereus.models.propagation import rtrsAcousticPropagationModel
from nereus.platform import TowedArrayPlatform
from nereus.plotter import plot_spectrogram, plot_world
from nereus.signal.anthropogenic import BroadbandShipSignal
from nereus.simulator import BroadbandPassiveSonarArraySimulator

# Set random seed for reproducibility
np.random.seed(1999)

# Shared plotting styles
BOUNDARY_LINE_STYLE = {
    "line_color": "black",
    "line_dash": "dash",
    "opacity": 0.5,
    "line_width": 1,
}
TARGET_FREQ_LINE_STYLE = {
    "line_color": "red",
    "line_dash": "dash",
    "opacity": 0.3,
    "line_width": 1.5,
}


def _real(signal: np.ndarray) -> np.ndarray:
    """Return a real-valued view for plotting/audio from complex or real input."""
    return np.real(signal) if np.iscomplexobj(signal) else np.asarray(signal)


def _timestep_boundaries(sim_params: dict) -> list[float]:
    """Return boundary times (seconds) between simulation timesteps."""
    dt = sim_params["time_interval"].total_seconds()
    return [i * dt for i in range(1, sim_params["num_steps"])]


def _add_vlines(
    fig: go.Figure,
    x_values: list[float],
    row: int | None = None,
    col: int | None = None,
    line_style: dict | None = None,
) -> None:
    """Add vertical marker lines to a Plotly figure or subplot."""
    style = dict(BOUNDARY_LINE_STYLE)
    if line_style:
        style.update(line_style)

    for x in x_values:
        kwargs = {"x": float(x), **style}
        if row is not None:
            kwargs["row"] = row
        if col is not None:
            kwargs["col"] = col
        fig.add_vline(**kwargs)


def _add_frequency_markers(
    fig: go.Figure,
    frequencies_hz: np.ndarray,
    row: int | None = None,
    col: int | None = None,
) -> None:
    """Mark reference frequencies on a Plotly figure."""
    _add_vlines(
        fig,
        [float(freq) for freq in frequencies_hz],
        row=row,
        col=col,
        line_style=TARGET_FREQ_LINE_STYLE,
    )


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


## 2. Define Simulation Parameters

Configure the simulation timing: 60 seconds total duration with 2-second timesteps (30 steps).

In [2]:
# Simulation parameters
SIM_RATE = 2.0
SIM_PARAMS = {
    "start_time": datetime.now().replace(hour=0, minute=0, second=0, microsecond=0),
    "time_interval": timedelta(seconds=SIM_RATE),
    "num_steps": 30,
}

total_duration_s = SIM_PARAMS["num_steps"] * SIM_PARAMS["time_interval"].total_seconds()

print("=== Broadband Continuous Signal Processing Demo ===")
print(f"Total simulation duration: {total_duration_s} s")
print(f"Number of timesteps: {SIM_PARAMS['num_steps']}")
print(f"Timestep interval: {SIM_PARAMS['time_interval'].total_seconds()} s")

=== Broadband Continuous Signal Processing Demo ===
Total simulation duration: 60.0 s
Number of timesteps: 30
Timestep interval: 2.0 s


## 3. Define Ship/Platform and Array Parameters

The ship platform remains stationary while the towed array (100 sensors, 200m cable, 1m spacing) is deployed at 200m depth.

In [3]:
SHIP_PARAMS = {
    "start_vector": np.array([0, 0, 0, 0, -10.0, 0]),  # Stationary ship
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3, 5],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)]
    ),
}

ARRAY_PARAMS = {
    "num_sensors": 100,
    "tow_cable_length": 200.0,
    "sensor_spacing": 1.0,
    "array_depth": -200.0,
}

print("Platform: Stationary ship at surface")
print(
    f"Array: {ARRAY_PARAMS['num_sensors']} sensors, "
    f"{ARRAY_PARAMS['tow_cable_length']}m cable, "
    f"{ARRAY_PARAMS['sensor_spacing']}m spacing"
)
print(f"Array depth: {ARRAY_PARAMS['array_depth']} m")

Platform: Stationary ship at surface
Array: 100 sensors, 200.0m cable, 1.0m spacing
Array depth: -200.0 m


## 4. Define Target Parameters

The target starts at 4.5 km range with slow motion. Its acoustic signature consists of 4 tonal components at different frequencies (60, 85, 120, 200 Hz) with amplitudes ranging from 75-100 dB re 1 µPa.

In [4]:
TARGET_PARAMS = {
    "start_vector": np.array([0, 0, 4500, -10, -10.0, 0]),  # Moving target
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3, 5],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0.001), ConstantVelocity(0.001), ConstantVelocity(0)]
    ),
    "amplitudes_upa": 10 ** (np.array([80.0, 95.0, 100.0, 75.0]) / 20),
    "frequencies_hz": np.array([60.0, 85.0, 120.0, 200.0]),
    "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
    "tonal_bandwidth_hz": 2.0,
    "noise_amplitude_upa": 10 ** (60 / 20),
    "noise_spectral_exponent": -1.0,
}

# Convert amplitudes to dB for display
amplitudes_db = 20 * np.log10(TARGET_PARAMS["amplitudes_upa"])

print(f"Target initial position: {TARGET_PARAMS['start_vector'][2]:.0f} m range")
print(f"Target depth: {TARGET_PARAMS['start_vector'][4]:.0f} m")
print(f"Tonal frequencies: {TARGET_PARAMS['frequencies_hz']} Hz")
print(f"Tonal amplitudes: {amplitudes_db} dB re 1 µPa")

Target initial position: 4500 m range
Target depth: -10 m
Tonal frequencies: [ 60.  85. 120. 200.] Hz
Tonal amplitudes: [ 80.  95. 100.  75.] dB re 1 µPa


## 5. Define Broadband Signal and Propagation Parameters

**Signal Parameters**: STFT processing with 500-sample frames (1 second at 500 Hz sampling rate) and 75% overlap (hop_factor=4).

**Propagation Parameters**: Constant sound speed profile and flat bathymetry at 100m depth.

In [5]:
SIGNAL_PARAMS = {
    "duration_s": total_duration_s,  # Long continuous signal
    "sampling_rate_hz": 500.0,
    "frame_len": 500,  # STFT frame length
    "hop_factor": 2,  # 75% overlap
    "fade_in_ms": 1000.0,  # Gentle fade-in
}

PROP_PARAMS = {
    # "ssp": Constant(speed=1500.0),
    "ssp": Linear(surface_speed=1500.0, gradient=0.2),
    # "ssp": Munk(),
    "attenuation_factor": 0.5,
    "bathymetry": FlatBathymetry(depth=-150.0),
    # "bathymetry": FlatBathymetry(depth=-5000.0),
    "step_m": 20.0,
    "azimuth_search_width": 2.0,
    "azimuth_resolution": 0.5,
    "elevation_range": (-25.0, 25.0),
    "elevation_resolution": 1.0,
}

# Sensor to analyze
SENSOR_TO_ANALYZE = ARRAY_PARAMS["num_sensors"] // 2

print(f"Signal sampling rate: {SIGNAL_PARAMS['sampling_rate_hz']} Hz")
print(f"STFT frame length: {SIGNAL_PARAMS['frame_len']} samples")
print(f"STFT hop factor: {SIGNAL_PARAMS['hop_factor']} (75% overlap)")
print(f"Bathymetry: Flat at {PROP_PARAMS['bathymetry'].depth} m depth")
print(f"Analyzing sensor: {SENSOR_TO_ANALYZE}")

Signal sampling rate: 500.0 Hz
STFT frame length: 500 samples
STFT hop factor: 2 (75% overlap)
Bathymetry: Flat at -150.0 m depth
Analyzing sensor: 50


## 6. Create Platform with Towed Array

Initialize the towed array platform and simulate motion through timesteps.

In [6]:
print("Creating platform with vertical motion...")
initial_state = GroundTruthState(SHIP_PARAMS["start_vector"], timestamp=SIM_PARAMS["start_time"])

platform = TowedArrayPlatform(
    states=[initial_state],
    position_mapping=SHIP_PARAMS["position_mapping"],
    velocity_mapping=SHIP_PARAMS["velocity_mapping"],
    transition_models=[SHIP_PARAMS["transition_model"]],
    transition_times=[timedelta(seconds=total_duration_s)],
    num_sensors=ARRAY_PARAMS["num_sensors"],
    cable_length_m=ARRAY_PARAMS["tow_cable_length"],
    sensor_spacing_m=ARRAY_PARAMS["sensor_spacing"],
    array_depth_m=ARRAY_PARAMS["array_depth"],
)

# Add vertical motion to platform
depth_change_per_step = -10.0  # Move up 10m per timestep
for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    platform.move(new_time)

# Get final array depth
final_platform_state = platform.get_platform_state_at(
    SIM_PARAMS["start_time"] + (SIM_PARAMS["num_steps"] - 1) * SIM_PARAMS["time_interval"]
)
final_depth = (
    final_platform_state.array.state_vector[2, 0]
    if final_platform_state
    else ARRAY_PARAMS["array_depth"]
)

print(f"  Array depth: {ARRAY_PARAMS['array_depth']} m -> {final_depth:.1f} m")

Creating platform with vertical motion...
  Array depth: -200.0 m -> -200.0 m


## 7. Create Target Trajectory

Generate the target ground truth path with acoustic metadata.

In [7]:
print("Creating target trajectory...")
target_states = [
    GroundTruthState(
        TARGET_PARAMS["start_vector"],
        timestamp=SIM_PARAMS["start_time"],
        metadata={
            "amplitudes_upa": TARGET_PARAMS["amplitudes_upa"],
            "frequencies_hz": TARGET_PARAMS["frequencies_hz"],
            "phases_rad": TARGET_PARAMS["phases_rad"],
            "position_mapping": TARGET_PARAMS["position_mapping"],
            "velocity_mapping": TARGET_PARAMS["velocity_mapping"],
        },
    )
]

transition_model = TARGET_PARAMS["transition_model"]
for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    time_interval = new_time - target_states[-1].timestamp
    new_state_vector = transition_model.function(
        target_states[-1], noise=False, time_interval=time_interval
    )
    new_state = GroundTruthState(
        new_state_vector,
        timestamp=new_time,
        metadata=target_states[-1].metadata,
    )
    target_states.append(new_state)

target_ground_truth = GroundTruthPath(target_states)

Creating target trajectory...


## 8. Create Broadband Signal Model

Use **BroadbandShipSignal** for realistic ship acoustics: broadband tonals (1 Hz bandwidth to simulate mechanical resonances) plus colored noise (85 dB re 1 µPa, pink noise spectrum with 1/f decay).

In [8]:
print("Creating broadband signal model...")

signal_model = BroadbandShipSignal(
    duration_s=SIGNAL_PARAMS["duration_s"],
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
    frame_len=SIGNAL_PARAMS["frame_len"],
    hop_factor=SIGNAL_PARAMS["hop_factor"],
    tonal_bandwidth_hz=1.0,  # Broader tonals (simulates mechanical resonances)
    noise_amplitude_upa=10 ** (85 / 20),  # 85 dB re 1 µPa background noise
    noise_spectral_exponent=-1.0,  # Pink noise (1/f)
    noise_freq_range_hz=(0.0, SIGNAL_PARAMS["sampling_rate_hz"]),
)

Creating broadband signal model...


## 9. Create Propagation Model and Simulator

Set up the **rtrs** (Range-dependent Two-dimensional Ray Simulation) acoustic propagation model and initialize the broadband passive sonar simulator.

In [9]:
prop_model = rtrsAcousticPropagationModel(
    ssp=PROP_PARAMS["ssp"],
    bathymetry=PROP_PARAMS["bathymetry"],
    step_m=PROP_PARAMS["step_m"],
    azimuth_search_width=PROP_PARAMS["azimuth_search_width"],
    azimuth_resolution=PROP_PARAMS["azimuth_resolution"],
    elevation_range=PROP_PARAMS["elevation_range"],
    elevation_resolution=PROP_PARAMS["elevation_resolution"],
)

signal_model = BroadbandShipSignal(
    duration_s=SIGNAL_PARAMS["duration_s"],
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
    frame_len=SIGNAL_PARAMS["frame_len"],
    hop_factor=SIGNAL_PARAMS["hop_factor"],
    tonal_bandwidth_hz=TARGET_PARAMS["tonal_bandwidth_hz"],
    noise_amplitude_upa=TARGET_PARAMS["noise_amplitude_upa"],
    noise_spectral_exponent=TARGET_PARAMS["noise_spectral_exponent"],
    noise_freq_range_hz=(0.0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    tonal_noise_is_constant=True,
    noise_is_constant=True,
)

simulator = BroadbandPassiveSonarArraySimulator(
    platform=platform,
    propagation_model=prop_model,
    signal_models=[signal_model],
    noise_model=None,
    beamformer=None,
    steering_calculator=None,
    ground_truth_paths=[target_ground_truth],
    fade_in_ms=SIGNAL_PARAMS["fade_in_ms"],
)

## 10. Run Simulation and Collect Sensor Data

Execute the simulation loop to generate sensor data at each timestep. The simulator processes signals in the frequency domain using STFT, applies propagation transfer functions, and reconstructs time-domain signals via overlap-add.

**Note**: This step may take several minutes depending on system performance.

In [10]:
print("Running broadband simulation...")
print()

all_sensor_signals = []

for _, sensor_data_set in simulator.sensor_data_gen():
    sensor_data = next(iter(sensor_data_set))
    all_sensor_signals.append(sensor_data.raw_signals)

# Concatenate all timestep signals
all_sensor_signals_array = np.concatenate(all_sensor_signals, axis=1)
continuous_signal = all_sensor_signals_array[SENSOR_TO_ANALYZE, :]
continuous_signal_first = all_sensor_signals_array[0, :]
continuous_signal_last = all_sensor_signals_array[-1, :]
time_axis = np.arange(continuous_signal.shape[0]) / SIGNAL_PARAMS["sampling_rate_hz"]

print("Simulation complete!")
print(f"  Total signal length: {len(continuous_signal)} samples")
print(f"  Signal duration: {len(continuous_signal) / SIGNAL_PARAMS['sampling_rate_hz']:.1f} s")


Running broadband simulation...

Simulation complete!
  Total signal length: 29500 samples
  Signal duration: 59.0 s


## 11. Retrieve Source Signal and Save WAV Files

Extract the source signal from the signal model cache and save both source and received signals as WAV files for audio playback.

In [11]:
print("Retrieving source signal from cache...")
try:
    source_signal = signal_model.get_source_signal()
except RuntimeError:
    print("Computing STFT to generate source signal...")
    signal_model.compute_stft(target_states[0])
    source_signal = signal_model.get_source_signal()

print(f"Source signal length: {len(source_signal)} samples")
print()
print("Displaying audio in notebook...")


def _audio_ready(
    signal: np.ndarray,
    sampling_rate_hz: float,
    min_playback_rate_hz: int = 8000,
) -> tuple[np.ndarray, int]:
    """Convert signal to normalized float32 and upsample for reliable browser playback."""
    signal_real = _real(signal).astype(np.float32)
    signal_real = np.nan_to_num(signal_real, nan=0.0, posinf=0.0, neginf=0.0)

    if signal_real.size == 0:
        return signal_real, int(sampling_rate_hz)

    # Robust scaling so sparse peaks don't make the rest effectively silent.
    scale = np.percentile(np.abs(signal_real), 99.5)
    if scale <= 0:
        scale = np.max(np.abs(signal_real))
    if scale > 0:
        signal_real = np.clip(signal_real / scale, -1.0, 1.0)

    playback_rate_hz = int(sampling_rate_hz)
    if 0 < playback_rate_hz < min_playback_rate_hz:
        up = int(np.ceil(min_playback_rate_hz / playback_rate_hz))
        signal_real = scipy_signal.resample_poly(signal_real, up=up, down=1).astype(np.float32)
        playback_rate_hz *= up

    return signal_real, playback_rate_hz


def _rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(x)))) if x.size else 0.0


source_audio, source_playback_rate = _audio_ready(source_signal, SIGNAL_PARAMS["sampling_rate_hz"])
received_audio, received_playback_rate = _audio_ready(
    continuous_signal, SIGNAL_PARAMS["sampling_rate_hz"]
)

print(
    f"Source playback: {len(source_audio)} samples @ {source_playback_rate} Hz, "
    f"peak={np.max(np.abs(source_audio)):.3f}, rms={_rms(source_audio):.3f}"
)
print(
    f"Received playback: {len(received_audio)} samples @ {received_playback_rate} Hz, "
    f"peak={np.max(np.abs(received_audio)):.3f}, rms={_rms(received_audio):.3f}"
)
print()

print("Source signal audio:")
display(Audio(data=source_audio, rate=source_playback_rate))

print("Received signal audio:")
display(Audio(data=received_audio, rate=received_playback_rate))


Retrieving source signal from cache...
Source signal length: 30000 samples

Displaying audio in notebook...
Source playback: 480000 samples @ 8000 Hz, peak=1.361, rms=0.353
Received playback: 472000 samples @ 8000 Hz, peak=1.429, rms=0.262

Source signal audio:


Received signal audio:


## 12. Visualise Time-Domain Signal and Spectrogram

Plot the continuous time-domain signal and its spectrogram. Red dashed lines mark timestep boundaries where the propagation transfer function is updated.

**Important**: The spectrogram uses proper PSD scaling (dB re 1 µPa²/Hz) for underwater acoustics.

In [12]:
boundary_times_s = _timestep_boundaries(SIM_PARAMS)
signal_real = _real(continuous_signal)

# Plot 1: Time-domain signal
fig_time = go.Figure()
fig_time.add_trace(
    go.Scatter(x=time_axis, y=signal_real, mode="lines", line=dict(width=1), opacity=0.7)
)
fig_time.update_yaxes(
    range=[np.percentile(signal_real, 1), np.percentile(signal_real, 99)],
    title_text="Pressure (µPa)",
)
fig_time.update_xaxes(title_text="Time (s)")
_add_vlines(fig_time, boundary_times_s)

fig_time.update_layout(
    title=f"Continuous Time-Domain Signal - Sensor {SENSOR_TO_ANALYZE}",
    template="plotly_white",
    height=400,
    width=900,
)
fig_time.show()

# Plot 2: Spectrogram (use plotter helper)
fig_spec = plot_spectrogram(
    signal_real,
    int(SIGNAL_PARAMS["sampling_rate_hz"]),
    n_fft=256,
    hop_length=64,
    y_lim=(0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    yaxis_format="hz",
    fig_size=(9, 4),
)
_add_vlines(fig_spec, boundary_times_s, line_style={"line_color": "grey"})
fig_spec.update_layout(title="Spectrogram - Broadband Continuous Signal")
fig_spec.show()


## 13. Visualise Target and Platform Trajectories

Show the trajectories in 2D Cartesian space (top view).

In [13]:
# Use plotter helper for world view
fig_world = plot_world(truths=[target_ground_truth], platform=platform)
fig_world.show()

## 14. Compare First and Last Sensor Signals

Visualize spatial variation: signals from opposite ends of the 100-sensor array show time delays and amplitude differences due to array geometry.

In [ ]:
boundary_times_s = _timestep_boundaries(SIM_PARAMS)
first_real = _real(continuous_signal_first)
last_real = _real(continuous_signal_last)

fig_end_sensors = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=("Sensor 0", f"Sensor {ARRAY_PARAMS['num_sensors'] - 1}"),
)

# First sensor
fig_end_sensors.add_trace(
    go.Scatter(
        x=time_axis,
        y=first_real,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        opacity=0.7,
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig_end_sensors.update_yaxes(
    range=[np.percentile(first_real, 1), np.percentile(first_real, 99)],
    title_text="Pressure (µPa)",
    row=1,
    col=1,
)

# Last sensor
fig_end_sensors.add_trace(
    go.Scatter(
        x=time_axis,
        y=last_real,
        mode="lines",
        line=dict(width=1, color="darkorange"),
        opacity=0.7,
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig_end_sensors.update_yaxes(
    range=[np.percentile(last_real, 1), np.percentile(last_real, 99)],
    title_text="Pressure (µPa)",
    row=2,
    col=1,
)
fig_end_sensors.update_xaxes(title_text="Time (s)", row=2, col=1)

for row in (1, 2):
    _add_vlines(fig_end_sensors, boundary_times_s, row=row, col=1)

fig_end_sensors.update_layout(
    title="Continuous Time-Domain Signals - Array End Sensors",
    template="plotly_white",
    height=760,
    width=900,
)
fig_end_sensors.show()

## 15. Analyse Source Signal

Examine the source signal before propagation:
- **Time domain**: Shows the temporal structure
- **Spectrogram**: Reveals frequency content evolution over time
- **Frequency spectrum**: Shows tonal peaks with proper amplitude scaling

**Critical FFT Scaling**: We multiply by `2.0/N` to convert FFT magnitude to single-sided amplitude spectrum in µPa.

In [15]:
source_time_axis = np.arange(len(source_signal)) / SIGNAL_PARAMS["sampling_rate_hz"]
source_real = _real(source_signal)

# Top: Source time domain
fig_src_time = go.Figure()
fig_src_time.add_trace(
    go.Scatter(
        x=source_time_axis,
        y=source_real,
        mode="lines",
        line=dict(width=1),
        opacity=0.8,
    )
)
fig_src_time.update_layout(
    title="Source Signal - Time Domain",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude (µPa)",
    template="plotly_white",
    height=400,
    width=900,
)
fig_src_time.show()

# Bottom: Source spectrogram (use plotter helper)
fig_src_spec = plot_spectrogram(
    source_real,
    int(SIGNAL_PARAMS["sampling_rate_hz"]),
    n_fft=256,
    hop_length=64,
    y_lim=(0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    yaxis_format="hz",
    fig_size=(9, 4),
)
fig_src_spec.update_layout(title="Source Signal - Spectrogram")
fig_src_spec.show()


## 16. Source Frequency Spectrum with Correct FFT Scaling

Plot the frequency spectrum of the source signal with **proper amplitude scaling**:
- FFT magnitude is multiplied by `2.0/N` to get single-sided amplitude
- Converted to dB re 1 µPa: `20 * log10(amplitude)`
- Red dashed lines mark target frequencies (60, 85, 120, 200 Hz)

In [16]:
signal_real_src = _real(source_signal)
freq_spectrum_src = np.fft.rfft(signal_real_src)
freq_axis_src = np.fft.rfftfreq(len(signal_real_src), 1 / SIGNAL_PARAMS["sampling_rate_hz"])

# FFT scaling: multiply by 2/N to get amplitude
N = len(signal_real_src)
spectrum_amplitude_upa = np.abs(freq_spectrum_src) * (2.0 / N)
spectrum_magnitude_db_src = 20 * np.log10(spectrum_amplitude_upa + 1e-10)

fig_src_freq = go.Figure()
fig_src_freq.add_trace(
    go.Scatter(x=freq_axis_src, y=spectrum_magnitude_db_src, mode="lines", line=dict(width=1))
)
_add_frequency_markers(fig_src_freq, TARGET_PARAMS["frequencies_hz"])

fig_src_freq.update_layout(
    title="Source Signal - Frequency Spectrum",
    xaxis_title="Frequency (Hz)",
    yaxis_title="Magnitude (dB re 1 µPa)",
    template="plotly_white",
    width=900,
    height=520,
)
fig_src_freq.show()

## 17. Compare Spectral Slices: Source vs Received

Extract 1-second spectral slices at the middle timestep (t=30s) and compare source and received signal spectra. This reveals propagation effects:

- **Transmission loss**: Reduction in amplitude at all frequencies
- **Frequency-dependent attenuation**: Higher frequencies may be attenuated more
- **Spectral distortion**: Changes in relative tonal amplitudes due to multipath interference

Both spectra use proper FFT scaling (`2.0/N` factor) for amplitude in dB re 1 µPa.

In [17]:
# Calculate middle timestep
middle_timestep_idx = SIM_PARAMS["num_steps"] // 2
timestep_duration_s = SIM_PARAMS["time_interval"].total_seconds()
middle_time_s = middle_timestep_idx * timestep_duration_s

# Extract 1-second slice around middle time
slice_duration_s = 1.0
slice_half_duration_s = slice_duration_s / 2
slice_start_time_s = middle_time_s - slice_half_duration_s
slice_end_time_s = middle_time_s + slice_half_duration_s

slice_start_sample = int(slice_start_time_s * SIGNAL_PARAMS["sampling_rate_hz"])
slice_end_sample = int(slice_end_time_s * SIGNAL_PARAMS["sampling_rate_hz"])

source_slice = source_signal[slice_start_sample:slice_end_sample]
received_slice = continuous_signal[slice_start_sample:slice_end_sample]

# Compute FFT with correct scaling
source_slice_real = _real(source_slice)
received_slice_real = _real(received_slice)

freq_spectrum_source_slice = np.fft.rfft(source_slice_real)
freq_spectrum_received_slice = np.fft.rfft(received_slice_real)
freq_axis_slice = np.fft.rfftfreq(len(source_slice_real), 1 / SIGNAL_PARAMS["sampling_rate_hz"])

N_slice = len(source_slice_real)
spectrum_amplitude_source_slice = np.abs(freq_spectrum_source_slice) * (2.0 / N_slice)
spectrum_amplitude_received_slice = np.abs(freq_spectrum_received_slice) * (2.0 / N_slice)

spectrum_magnitude_db_source_slice = 20 * np.log10(spectrum_amplitude_source_slice + 1e-10)
spectrum_magnitude_db_received_slice = 20 * np.log10(spectrum_amplitude_received_slice + 1e-10)

fig_slice_compare = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        f"Source Signal - Spectral Slice at t={middle_time_s:.1f}s (±{slice_half_duration_s}s)",
        f"Received Signal - Spectral Slice at t={middle_time_s:.1f}s (±{slice_half_duration_s}s)",
    ),
)

# Source slice
fig_slice_compare.add_trace(
    go.Scatter(
        x=freq_axis_slice,
        y=spectrum_magnitude_db_source_slice,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig_slice_compare.update_yaxes(title_text="Magnitude (dB re 1 µPa)", row=1, col=1)
fig_slice_compare.update_xaxes(range=[0, 250], row=1, col=1)
_add_frequency_markers(fig_slice_compare, TARGET_PARAMS["frequencies_hz"], row=1, col=1)

# Received slice
fig_slice_compare.add_trace(
    go.Scatter(
        x=freq_axis_slice,
        y=spectrum_magnitude_db_received_slice,
        mode="lines",
        line=dict(width=1, color="darkorange"),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig_slice_compare.update_yaxes(title_text="Magnitude (dB re 1 µPa)", row=2, col=1)
fig_slice_compare.update_xaxes(title_text="Frequency (Hz)", range=[0, 250], row=2, col=1)
_add_frequency_markers(fig_slice_compare, TARGET_PARAMS["frequencies_hz"], row=2, col=1)

fig_slice_compare.update_layout(
    title="Spectral Slice Comparison - Source vs Received at Middle Timestep",
    template="plotly_white",
    height=760,
    width=900,
)
fig_slice_compare.show()